# Search problems

_Artificial Intelligence (CS221)_

**Start somewhere, take actions that cost something, reach a goal as cheaply as possible.**

Some problems need planning, not a reflex. You must take a sequence of actions to reach a goal.

     A search problem is a clean way to describe these. Think of finding a path through a maze.

---

This notebook builds a search problem from scratch, one piece at a time. We'll model a small graph, run breadth-first search (BFS) to find a shortest path, then watch the same idea route a courier across a city map. Run each cell, read the note above it, and don't be afraid to tweak the numbers. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — Python

### Step 1 — Define the search problem

A search problem needs three things: a **start** state, a **goal** state, and a **successor** function that says which states you can move to from any given state. Here the states are nodes `S, A, B, ... , G`, and `succ[u]` lists the nodes reachable in one step from `u`. This little graph is our maze: every path from `S` is a sequence of actions, and we're after the cheapest one (fewest steps) that lands on `G`.

In [ ]:
from collections import deque

# succ[u] = the states reachable from u in one action (an edge of the graph).
succ = {
    "S": ["A", "B"],
    "A": ["C", "D"],
    "B": ["D", "E"],
    "C": ["G"],
    "D": ["G"],
    "E": ["G"],
    "G": [],
}

start = "S"
goal = "G"

### Step 2 — Explore the graph with BFS

Breadth-first search explores states in **layers**: all states one step from the start, then all states two steps away, and so on. We keep a **frontier** (a FIFO queue) of states waiting to be expanded and always pop from the *front*, so nearer states come out first. Because every edge here costs the same, the first time BFS reaches the goal it has found a path with the fewest edges. As we discover each new state `v` we record which state `u` we came from in `parent`, so we can reconstruct the path afterwards.

In [ ]:
frontier = deque([start])   # states waiting to be expanded (FIFO -> BFS)
parent = {start: None}      # parent[v] = the state we reached v from
order = []                  # the order in which we expand states

while frontier:
    u = frontier.popleft()      # BFS: take from the front of the queue
    order.append(u)

    if u == goal:
        break                   # stop as soon as we reach the goal

    for v in succ[u]:
        if v not in parent:     # first time we've seen v
            parent[v] = u
            frontier.append(v)

### Step 3 — Rebuild and read the path

BFS found the goal, but the path is stored *backwards* in `parent`: each state points to the one before it. To recover the route we start at the goal and follow the `parent` links back to the start, then reverse the list. The number of edges (one less than the number of states on the path) is the cost of this shortest path.

In [ ]:
path = []
node = goal

while node is not None:          # walk parent links from goal back to start
    path.append(node)
    node = parent[node]

path.reverse()                   # flip it so it reads start -> goal

num_edges = len(path) - 1
print("discovery order:", order)
print("path S -> G     :", " -> ".join(path), "(length", num_edges, ")")

## Visualize the data & results

_What route does BFS find for a courier driving from the Ferry Building to the Castro?_

### Step 1 — Build the city map as a graph

Now we apply the exact same search idea to a real-ish map of San Francisco districts. Each district is a node and each road is an **edge**. Roads go both ways, so for every edge `(u, v)` we record `v` as a neighbour of `u` *and* `u` as a neighbour of `v`. The result, `nbr`, is the successor function for this problem.

In [ ]:
from collections import deque
import matplotlib.pyplot as plt

# Each pair is a two-way road between two districts.
edges = [
    ("FB", "US"), ("FB", "CT"), ("CT", "NB"), ("US", "CT"), ("US", "SO"),
    ("SO", "MS"), ("US", "CC"), ("CC", "MS"), ("MS", "CA"), ("CC", "CA"), ("NB", "US"),
]

# Build the neighbour (successor) map. Roads are undirected, so add both directions.
nbr = {}
for u, v in edges:
    nbr.setdefault(u, []).append(v)
    nbr.setdefault(v, []).append(u)

### Step 2 — Run BFS from the Ferry Building to the Castro

Same BFS loop as before, just on the bigger map: start at `FB` (Ferry Building) and search for `CA` (the Castro). We expand neighbours in sorted order so the search is deterministic, record each state's parent, and then follow the parent links back to recover the route.

In [ ]:
start, goal = "FB", "CA"
parent = {start: None}
frontier = deque([start])

while frontier:
    u = frontier.popleft()
    if u == goal:
        break
    for v in sorted(nbr[u]):        # sorted -> deterministic exploration
        if v not in parent:
            parent[v] = u
            frontier.append(v)

path = []
node = goal
while node is not None:
    path.append(node)
    node = parent[node]
path.reverse()

print("route", path)

### Step 3 — Draw the map and highlight the route

Finally we plot the districts at hand-picked coordinates and draw the roads between them. The BFS route is overlaid in orange, and every district on that route is coloured green while the rest stay blue. This makes it easy to see that BFS picked a path with as few hops as possible from the Ferry Building to the Castro.

In [ ]:
# Hand-placed screen coordinates for each district.
pos = {
    "FB": (540, 300), "US": (360, 210), "CT": (420, 330), "NB": (470, 420),
    "CC": (200, 150), "SO": (330, 90), "MS": (230, 40), "CA": (90, 90),
}

fig, ax = plt.subplots(figsize=(8, 6))

# Draw every road as a grey line.
for u, v in edges:
    ax.plot([pos[u][0], pos[v][0]], [pos[u][1], pos[v][1]], color="#888", zorder=1)

# Overlay the BFS route in orange.
route_x = [pos[n][0] for n in path]
route_y = [pos[n][1] for n in path]
ax.plot(route_x, route_y, color="#ffb454", lw=3, zorder=2)

# Draw each district: green if it's on the route, blue otherwise.
on_path = set(path)
for name, (x, y) in pos.items():
    color = "#7ee787" if name in on_path else "#4ea1ff"
    ax.scatter(x, y, s=400, zorder=3, c=color)
    ax.annotate(name, (x, y), ha="center", va="center")

ax.set_title("SF district map: BFS route FB to CA")
plt.show()